In [1]:
""" 0. set-up part:  import necessary libraries and set up environment """

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize
from collections import Counter, defaultdict
import numpy as np
import math
import copy
import itertools
import matplotlib.pyplot as plt
import matplotlib as mpl

import joblib
from joblib import Parallel, delayed
from threading import Thread

import os
import pickle
import time

import operator
from functools import reduce
import json
import cProfile

import gensim
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

import tomotopy as tp

# download nltk data once time
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('averaged_perceptron_tagger')
# nltk.download('omw-1.4')
# nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')

#  chinese character support in matplotlib
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS' 'SimHei' 'DejaVu Sans']  
plt.rcParams['axes.unicode_minus'] = False

In [2]:
""" 1.1 Data Preprocessing: load data, clean text, lemmatization, remove low-frequency words"""

# Map POS tags to WordNet format， Penn Treebank annotation: fine-grained (45 tags), WordNet annotation: coarse-grained (4 tags: a, v, n, r)
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return 'a'  # 形容词
    elif treebank_tag.startswith('V'):
        return 'v'  # 动词
    elif treebank_tag.startswith('N'):
        return 'n'  # 名词
    elif treebank_tag.startswith('R'):
        return 'r'  # 副词
    else:
        return 'n'  # 默认名词

# Text cleaning and lemmatization preprocessing function
def clean_and_lemmatize(text):
    if pd.isnull(text):
        return []
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # Remove non-alphabetic characters using regex
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    pos_tags = pos_tag(tokens)
    lemmatized = [lemmatizer.lemmatize(w, get_wordnet_pos(pos)) for w, pos in pos_tags]
    return lemmatized  

#-----------------Load data----------------
data = pd.read_excel('./data/raw/papers_CM.xlsx', usecols=['PaperID', 'Abstract', 'Keywords', 'Year'])

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# clean and lemmatize the abstracts
data['Lemmatized_Tokens'] = data['Abstract'].apply(clean_and_lemmatize)

# count word frequencies
all_tokens = [word for tokens in data['Lemmatized_Tokens'] for word in tokens]
word_counts = Counter(all_tokens)

# set a minimum frequency threshold for valid words
min_freq = 10
valid_words = set([word for word, freq in word_counts.items() if freq >= min_freq])

# remove rare words based on frequency threshold
def remove_rare_words(tokens):
    return [word for word in tokens if word in valid_words]

data['Filtered_Tokens'] = data['Lemmatized_Tokens'].apply(remove_rare_words)

# join tokens back into cleaned abstracts
data['Cleaned_Abstract'] = data['Filtered_Tokens'].apply(lambda x: " ".join(x))

# create a cleaned DataFrame with relevant columns
cleaned_data = data[['PaperID', 'Year', 'Cleaned_Abstract']]
cleaned_data = cleaned_data[~(cleaned_data['PaperID'] == 57188)] # this paper has no abstract
cleaned_data = cleaned_data.reset_index(drop=True) 
cleaned_data.insert(0, 'Document_ID', range(len(cleaned_data))) 
abstract_list = cleaned_data['Cleaned_Abstract'].apply(lambda x: x.split()).tolist()

corpus = {doc_id: abstract_list for doc_id, abstract_list in enumerate(abstract_list)}
# cleaned_data.to_csv('./data/processed/cleaned_data.xlsx', index=False, encoding='utf-8-sig')

In [3]:
# ===== Enhanced Coherence Calculation Function (Supports Multiple Metrics Including NPMI) =====
def calculate_multiple_coherence_metrics(mdl, corpus_docs, metrics=['c_v', 'c_npmi'], fast_mode=True, top_n=5):
    """
    Enhanced coherence calculation function - supports multiple coherence metrics including NPMI
    
    Supported coherence metrics:
    - c_v: Vector space-based coherence (default)
    - c_npmi: Normalized Pointwise Mutual Information (NPMI)
    
    Supported model types:
    - Single-layer models: LDA, CTM
    - Hierarchical models: hLDA, PAM, hPAM
    - Non-parametric models: HDP
    
    Returns: dict containing various coherence metrics
    """
    try:
        docs = list(corpus_docs)
        dictionary = Dictionary(docs)
        
        # Check if it is a hierarchical model
        model_type_str = str(type(mdl))
        is_hierarchical = hasattr(mdl, 'depth') or 'HLDA' in model_type_str or 'PAM' in model_type_str
        
        if is_hierarchical:
            # Hierarchical model: calculate coherence by level, weighted average
            metrics_results = {metric: [] for metric in metrics}
            layer_weights = []
            
            try:
                for level in range(getattr(mdl, 'depth', 3)):
                    level_topics = []
                    level_doc_count = 0
                    
                    # Iterate through all nodes of this level
                    for k in range(getattr(mdl, 'k', 100)):
                        try:
                            topic_words = mdl.get_topic_words(k, top_n=top_n)
                            if topic_words:
                                words = [word for word, prob in topic_words]
                                level_topics.append(words)
                                level_doc_count += 1
                        except:
                            continue
                    
                    if level_topics:
                        # Calculate coherence for each metric
                        for metric in metrics:
                            try:
                                cm = CoherenceModel(
                                    topics=level_topics,
                                    texts=docs,
                                    dictionary=dictionary,
                                    coherence=metric,
                                    processes=1
                                )
                                score = cm.get_coherence()
                                if score and not math.isnan(score):
                                    metrics_results[metric].append(score)
                                else:
                                    metrics_results[metric].append(0.1)
                            except Exception as e:
                                print(f"Hierarchical model {metric} calculation warning: {e}")
                                metrics_results[metric].append(0.1)
                        
                        layer_weights.append(level_doc_count)
            except:
                pass
            
            # Calculate weighted average
            final_results = {}
            for metric in metrics:
                if metrics_results[metric] and layer_weights:
                    total_weight = sum(layer_weights)
                    if total_weight > 0:
                        weighted_avg = sum(score * w for score, w in zip(metrics_results[metric], layer_weights)) / total_weight
                        final_results[metric] = weighted_avg
                    else:
                        final_results[metric] = 0.1
                else:
                    final_results[metric] = 0.1
            
            return final_results
        
        # Single-layer model: calculate coherence for all topics
        num_topics = getattr(mdl, 'k', None) or getattr(mdl, 'num_topics', None) or 100
        topics = []
        
        for k in range(num_topics):
            try:
                topic_words = mdl.get_topic_words(k, top_n=top_n)
                if topic_words:
                    words = [word for word, prob in topic_words]
                    topics.append(words)
            except:
                continue
        
        if not topics:
            return {metric: 0.1 for metric in metrics}
        
        # Calculate coherence for each metric
        final_results = {}
        for metric in metrics:
            try:
                cm = CoherenceModel(
                    topics=topics,
                    texts=docs,
                    dictionary=dictionary,
                    coherence=metric,
                    processes=1
                )
                score = cm.get_coherence()
                final_results[metric] = score if score and not math.isnan(score) else 0.1
            except Exception as e:
                print(f"Single-layer model {metric} calculation warning: {e}")
                final_results[metric] = 0.1
        
        return final_results
        
    except Exception as e:
        print(f"Coherence calculation error: {e}")
        return {metric: 0.1 for metric in metrics}

In [4]:
def calculate_renyi_entropy_unweighted(model, alpha=2):
    """
    Calculate the Renyi entropy for all topics (unweighted version, direct average)
    - model: topic model object (e.g., tomotopy LDA/CTM/PAM/hLDA)
    - alpha: order of Renyi entropy (commonly 2)
    Returns: the average of Renyi entropies for all topics
    """
    # ...existing code...
    import numpy as np
    entropies = []
    num_topics = getattr(model, 'k', None) or getattr(model, 'num_topics', None) or 0
    for k in range(num_topics):
        try:
            topic_probs = np.array([prob for word, prob in model.get_topic_words(k, top_n=-1)])
            topic_probs = topic_probs / topic_probs.sum()
            if len(topic_probs) > 0:
                renyi = (1/(1-alpha)) * np.log(np.sum(topic_probs**alpha))
                entropies.append(renyi)
            else:
                entropies.append(0)
        except:
            entropies.append(0)
    if entropies:
        return float(np.mean(entropies))
    else:
        return 0.0

In [5]:
# ===== 加权Renyi熵计算函数 (Weighted Renyi Entropy) =====
def get_topic_doc_counts(model, threshold=0.01):
    """
    获取每个主题覆盖的文档数（即每个主题出现在了多少个文档中）
    """
    num_topics = getattr(model, 'k', None) or getattr(model, 'num_topics', None) or 0
    topic_doc_counts = [0] * num_topics
    for doc in model.docs:
        try:
            topic_dist = doc.get_topic_dist()
        except:
            continue
        for k, prob in enumerate(topic_dist):
            if prob > threshold:
                topic_doc_counts[k] += 1
    return topic_doc_counts

def calculate_weighted_renyi_entropy(model, alpha=2):
    """
    按主题覆盖的文档数加权的Renyi熵
    """
    entropies = []
    num_topics = getattr(model, 'k', None) or getattr(model, 'num_topics', None) or 0
    for k in range(num_topics):
        try:
            topic_probs = np.array([prob for word, prob in model.get_topic_words(k, top_n=-1)])
            if topic_probs.sum() > 0:
                topic_probs = topic_probs / topic_probs.sum()
                renyi = (1/(1-alpha)) * np.log(np.sum(topic_probs**alpha))
                entropies.append(renyi)
            else:
                entropies.append(0)
        except:
            entropies.append(0)
    
    # 按主题的文档覆盖数进行加权
    topic_doc_counts = get_topic_doc_counts(model)
    total_docs_covered = sum(topic_doc_counts)
    
    if total_docs_covered > 0 and len(entropies) == len(topic_doc_counts):
        return np.average(entropies, weights=topic_doc_counts)
    elif entropies:
        return np.mean(entropies) # 如果加权失败，则返回普通平均值
    else:
        return 0.0

In [6]:
from scipy.spatial.distance import jensenshannon
import numpy as np

def calculate_topic_diversity_jsd(model, top_n=25):
    """
    计算所有主题对之间的平均JSD（Jensen-Shannon Divergence），以衡量主题多样性。
    JSD值域为[0, 1]，值越高代表主题间差异越大，多样性越好。

    - model: 训练好的tomotopy模型。
    - top_n: 用于计算JSD的每个主题的top N个词。
    """
    try:
        num_topics = getattr(model, 'k', None) or getattr(model, 'num_topics', None) or 0
        if num_topics < 2:
            return 0.0

        # 1. 收集所有主题的top N词和概率，并建立一个共享词汇表
        topic_word_probs = []
        vocab = set()
        for k in range(num_topics):
            try:
                words = model.get_topic_words(k, top_n=top_n)
                if not words: continue
                topic_word_probs.append(dict(words))
                vocab.update([word for word, prob in words])
            except:
                continue
        
        if len(topic_word_probs) < 2:
            return 0.0

        vocab_list = sorted(list(vocab))
        vocab_map = {word: i for i, word in enumerate(vocab_list)}
        
        # 2. 将每个主题的词分布转换为对齐的概率向量
        aligned_probs = []
        for topic_dict in topic_word_probs:
            prob_vector = np.zeros(len(vocab_list))
            for word, prob in topic_dict.items():
                if word in vocab_map:
                    prob_vector[vocab_map[word]] = prob
            
            # 归一化，使其成为有效的概率分布
            if prob_vector.sum() > 1e-9:
                prob_vector /= prob_vector.sum()
            else:
                continue # 跳过无效的空主题
            aligned_probs.append(prob_vector)

        if len(aligned_probs) < 2:
            return 0.0

        # 3. 计算所有主题对之间的JSD
        jsd_values = []
        for i in range(len(aligned_probs)):
            for j in range(i + 1, len(aligned_probs)):
                p = aligned_probs[i]
                q = aligned_probs[j]
                jsd = jensenshannon(p, q, base=2)
                if not np.isnan(jsd):
                    jsd_values.append(jsd**2) # JSD距离通常使用JSD值的平方

        if not jsd_values:
            return 0.0
        
        return float(np.mean(jsd_values))

    except Exception as e:
        # print(f"Error calculating JSD: {e}")
        return 0.0

In [7]:
def tune_ctm_model(docs, k_range, eta_range, seed=42, max_iters=1000):
    """
    Performs a grid search for the CTM model by iterating through K and Eta.
    It calls separate functions to calculate metrics for each trained model.
    """
    tuning_results = []
    best_npmi_score = -1
    best_params = {}
    best_model = None
    
    total_runs = len(k_range) * len(eta_range)
    print(f"🚀 Starting CTM Grid Search for {total_runs} combinations...")
    
    for i, (k, eta) in enumerate(itertools.product(k_range, eta_range), 1):
        print(f"\n--- Run {i}/{total_runs}: Testing K={k}, Eta={eta} ---")
        start_time = time.time()
        
        try:
            # 1. Create and train the model
            model = tp.CTModel(k=k, eta=eta, smoothing_alpha=ALPHA, seed=seed)
            for doc in docs:
                model.add_doc(doc)
            
            model.train(0)
            print(f"Starting training for K={k}, Eta={eta}...")
            for _ in range(0, max_iters, 100):
                model.train(100)
            print("Training finished.")

            # 2. Call metric functions to evaluate
            print("Calculating metrics...")
            
            perplexity = math.exp(-model.ll_per_word)
            coherence_scores = calculate_multiple_coherence_metrics(model, docs, metrics=['c_v', 'c_npmi'])
            unweighted_entropy = calculate_renyi_entropy_unweighted(model, alpha=2)
            weighted_entropy = calculate_weighted_renyi_entropy(model, alpha=2)
            jsd_diversity = calculate_topic_diversity_jsd(model, top_n=25)
            
            # 3. Store results
            tuning_results.append({
                'K': k, 'Eta': eta, 
                'NPMI': coherence_scores.get('c_npmi', 0), 
                'C_v': coherence_scores.get('c_v', 0), 
                'Perplexity': perplexity, 
                'Entropy_Unweighted': unweighted_entropy,
                'Entropy_Weighted': weighted_entropy,
                'JSD_Diversity': jsd_diversity,
                'Time_sec': time.time() - start_time
            })
            
            print(f"   => Results: PPL={perplexity:.2f}, CV={coherence_scores.get('c_v', 0):.4f}, NPMI={coherence_scores.get('c_npmi', 0):.4f}, JSD={jsd_diversity:.4f}, Entropy_W={weighted_entropy:.4f}, Entropy_UW={unweighted_entropy:.4f}")
            # 4. Track best model
            if coherence_scores.get('c_npmi', 0) > best_npmi_score:
                best_npmi_score = coherence_scores.get('c_npmi', 0)
                best_params = {'K': k, 'Eta': eta}
                best_model = model
                print(f"   ✨ New best model found!")

        except Exception as e:
            print(f"   ❌ Run failed for K={k}, Eta={eta}: {e}")

    results_df = pd.DataFrame(tuning_results)
    print("\n\n✅ CTM Tuning Complete!")
    print("--- Tuning Results Summary ---")
    print(results_df.to_string())
    
    if best_params:
        print(f"\n🏆 Best Parameters (by NPMI): K={best_params['K']}, Eta={best_params['Eta']} with NPMI score of {best_npmi_score:.4f}")
    else:
        print("\n❌ No successful runs completed.")

    return results_df, best_model, best_params

In [8]:
# ===== General Topic Analysis Function =====
def analyze_model_topics(model, model_name="Model", top_words=5, min_prob=0.01, max_display=10):
    """
    General topic analysis function - applicable to all topic models
    
    Functionality:
    - Extracts active topics
    - Displays topic words and weights
    - Calculates topic activity rate
    """
    print(f"\n🔍 {model_name} Topic Analysis (showing top {top_words} words):")
    print("=" * 80)
    
    active_topics = 0
    topic_info = []
    
    num_topics = getattr(model, 'k', None) or getattr(model, 'num_topics', None) or 0
    
    for k in range(num_topics):
        try:
            topic_words = model.get_topic_words(k, top_n=top_words)
            if topic_words and topic_words[0][1] > min_prob:
                active_topics += 1
                words_str = ", ".join([f"{word}({prob:.3f})" for word, prob in topic_words[:5]])
                topic_info.append((k, topic_words[0][1], words_str))
                
                if active_topics <= max_display:
                    print(f"Topic {k:3d} (weight:{topic_words[0][1]:.3f}): {words_str}")
        except:
            continue
    
    if active_topics > max_display:
        print(f"... (and {active_topics - max_display} more active topics)")
    
    print(f"\n📊 {model_name} Topic Statistics:")
    print(f"   - Active topics: {active_topics}/{num_topics}")
    print(f"   - Topic activity rate: {active_topics/num_topics*100:.1f}%")
    
    return topic_info

In [9]:
def calculate_r_hat(chains_history):
    """
    根据多个MCMC链的后半部分历史记录计算R-hat（Gelman-Rubin诊断）值。
    
    参数:
    - chains_history (np.ndarray): 一个2D numpy数组，形状为 (M, N)，
      其中 M 是链的数量（运行次数），N 是每个链记录的迭代次数。
      数组中的值是对数似然（log-likelihood）。

    返回:
    - float: R-hat值。如果无法计算，则返回 np.nan。
    """
    # 仅使用后半部分的样本进行计算，这是标准做法
    num_iterations = chains_history.shape[1]
    start_index = num_iterations // 2
    
    if chains_history.shape[0] < 2 or (num_iterations - start_index) < 2:
        return np.nan

    chains_history = chains_history[:, start_index:]
    num_chains, num_used_iterations = chains_history.shape
    
    # 1. 计算每个链的均值
    chain_means = np.mean(chains_history, axis=1)
    
    # 2. 计算每个链的方差
    chain_variances = np.var(chains_history, axis=1, ddof=1)
    
    # 3. 计算链内方差的均值 (W)
    W = np.mean(chain_variances)
    
    # 4. 计算链间方差 (B)
    overall_mean = np.mean(chain_means)
    B = num_used_iterations / (num_chains - 1) * np.sum((chain_means - overall_mean)**2)
    
    # 5. 估计目标分布的方差 (Var_hat)
    var_hat = (1 - 1 / num_used_iterations) * W + (1 / num_used_iterations) * B
    
    if W == 0:
        return np.nan
        
    # 6. 计算R-hat
    r_hat = np.sqrt(var_hat / W)
    
    return r_hat

In [10]:
def calculate_weighted_coherence(model, corpus_docs, metrics=['c_v', 'c_npmi'], top_n=10, threshold=0.01):
    """
    计算按主题的文档覆盖率加权的 coherence 分数 (NPMI, C_v)。
    
    Args:
        model: 训练好的 tomotopy 模型。
        corpus_docs: 用于计算 coherence 的原始文档列表。
        metrics: 要计算的指标列表。
        top_n: 用于定义主题的 top N 个词。
        threshold: 判断一个主题在文档中是否“显著”的概率阈值。

    Returns:
        一个包含加权和未加权 coherence 分数的字典。
    """
    try:
        docs = list(corpus_docs)
        dictionary = Dictionary(docs)
        num_topics = getattr(model, 'k', 0) or getattr(model, 'num_topics', 0)
        
        if num_topics == 0:
            return {}

        # 1. 提取所有主题的 top words
        topics = []
        for k in range(num_topics):
            topic_words = [word for word, prob in model.get_topic_words(k, top_n=top_n)]
            if topic_words:
                topics.append(topic_words)
        
        if not topics:
            return {}

        # 2. 获取每个主题的权重 (文档覆盖数)
        topic_doc_counts = np.zeros(num_topics)
        for doc in model.docs:
            topic_dist = doc.get_topic_dist()
            for k, prob in enumerate(topic_dist):
                if prob > threshold:
                    topic_doc_counts[k] += 1
        
        total_docs_covered = np.sum(topic_doc_counts)
        
        results = {}
        for metric in metrics:
            try:
                cm = CoherenceModel(
                    topics=topics,
                    texts=docs,
                    dictionary=dictionary,
                    coherence=metric,
                    processes=1
                )
                
                # 获取每个主题的分数
                per_topic_scores = cm.get_coherence_per_topic()
                
                # 计算简单平均值 (未加权)
                unweighted_avg = np.mean(per_topic_scores) if per_topic_scores else 0.0
                results[f'unweighted_{metric}'] = unweighted_avg
                
                # 计算加权平均值
                if total_docs_covered > 0 and per_topic_scores:
                    weighted_avg = np.average(per_topic_scores, weights=topic_doc_counts[:len(per_topic_scores)])
                    results[f'weighted_{metric}'] = weighted_avg
                else:
                    results[f'weighted_{metric}'] = unweighted_avg # 如果没有权重，则退回到简单平均
            except Exception:
                results[f'unweighted_{metric}'] = 0.0
                results[f'weighted_{metric}'] = 0.0

        return results

    except Exception as e:
        print(f"计算加权Coherence时出错: {e}")
        return {}

In [11]:
def tune_ctm_model(docs, k_range, eta_range, ALPHA, num_runs=3, seed=42, max_iters=1000, **kwargs):
    """
    Performs a grid search for CTM, recording the results of each individual run.
    Uses an early stopping criterion based on log-likelihood improvement.
    Calculates a comprehensive set of metrics including weighted/unweighted coherence, entropy, and JSD.
    """
    all_run_results = []
    best_single_run_npmi = -1
    best_params = {}
    best_model = None

    # --- 收敛判断参数 ---
    check_interval = 10
    consecutive_checks_limit = 10
    improvement_threshold = 0.01
    # -------------------------

    total_combinations = len(k_range) * len(eta_range)
    print(f"🚀 Starting CTM Grid Search for {total_combinations} combinations, {num_runs} runs each...")
    print(f"   (Convergence check: LL improvement < {improvement_threshold*100}% for {consecutive_checks_limit} consecutive checks of {check_interval} iterations)")

    for i, (k, eta) in enumerate(itertools.product(k_range, eta_range), 1):
        print(f"\n--- Combination {i}/{total_combinations}: Testing K={k}, Eta={eta} for {num_runs} runs ---")
        
        for run_idx in range(num_runs):
            start_time = time.time()
            current_seed = seed + run_idx
            print(f"  - Run {run_idx + 1}/{num_runs} (seed={current_seed})...")
            
            try:
                model = tp.CTModel(k=k, eta=eta, smoothing_alpha=ALPHA, seed=current_seed)
                for doc in docs:
                    model.add_doc(doc)
                
                # 1. 带有提前停止逻辑的训练循环
                ll_history = []
                consecutive_small_improvements = 0
                convergence_status = "Max Iterations Reached"
                
                burn_in_iters = 200
                model.train(burn_in_iters)
                ll_history.append(model.ll_per_word)
                
                for current_iter in range(burn_in_iters, max_iters, check_interval):
                    model.train(check_interval)
                    current_ll = model.ll_per_word
                    prev_ll = ll_history[-1]
                    
                    if prev_ll != 0 and prev_ll is not None and current_ll is not None:
                        improvement = (current_ll - prev_ll) / abs(prev_ll)
                    else:
                        improvement = float('inf')

                    if improvement < improvement_threshold:
                        consecutive_small_improvements += 1
                    else:
                        consecutive_small_improvements = 0
                    
                    ll_history.append(current_ll)

                    if consecutive_small_improvements >= consecutive_checks_limit:
                        convergence_status = "Converged"
                        print(f"    => {convergence_status} after {current_iter + check_interval} iterations.")
                        break
                else:
                    print(f"    => {convergence_status} at {max_iters} iterations.")

                # 2. 评估并记录单次运行的指标
                perplexity = math.exp(-model.ll_per_word) if model.ll_per_word is not None else float('inf')
                
                # [核心修改] 调用所有新的指标函数
                coherence_scores = calculate_weighted_coherence(model, docs, metrics=['c_v', 'c_npmi'])
                unweighted_entropy = calculate_renyi_entropy_unweighted(model, alpha=2)
                weighted_entropy = calculate_weighted_renyi_entropy(model, alpha=2)
                jsd_diversity = calculate_topic_diversity_jsd(model, top_n=25)
                
                # [核心修改] 扩展 run_result 字典
                run_result = {
                    'K': k,
                    'Eta': eta,
                    'Run_Index': run_idx + 1,
                    'Perplexity': perplexity,
                    'Weighted_NPMI': coherence_scores.get('weighted_c_npmi', 0),
                    'Weighted_Cv': coherence_scores.get('weighted_c_v', 0),
                    'Unweighted_NPMI': coherence_scores.get('unweighted_c_npmi', 0),
                    'Unweighted_Cv': coherence_scores.get('unweighted_c_v', 0),
                    'Weighted_Renyi_Entropy': weighted_entropy,
                    'Unweighted_Renyi_Entropy': unweighted_entropy,
                    'JSD_Diversity': jsd_diversity,
                    'Iterations': model.global_step,
                    'Convergence_Status': convergence_status,
                    'Time_sec': time.time() - start_time
                }
                all_run_results.append(run_result)
                
                # [核心修改] 更新打印信息
                print(f"    => Results: PPL={run_result['Perplexity']:.2f}, W_NPMI={run_result['Weighted_NPMI']:.4f}, JSD={run_result['JSD_Diversity']:.4f}, Iters={run_result['Iterations']:.0f}")

                # 3. 追踪最佳模型（基于单次运行的最佳加权NPMI）
                if run_result['Weighted_NPMI'] > best_single_run_npmi:
                    best_single_run_npmi = run_result['Weighted_NPMI']
                    best_params = {'K': k, 'Eta': eta}
                    best_model = model

            except Exception as e:
                print(f"    ❌ Run {run_idx + 1} failed for K={k}, Eta={eta}: {e}")

    results_df = pd.DataFrame(all_run_results)
    print("\n\n✅ CTM Tuning Complete!")
    
    # [核心修改] 更新结果表格的列顺序
    if not results_df.empty:
        cols_order = [
            'K', 'Eta', 'Run_Index', 'Weighted_NPMI', 'Perplexity', 'JSD_Diversity',
            'Weighted_Cv', 'Unweighted_NPMI', 'Unweighted_Cv',
            'Weighted_Renyi_Entropy', 'Unweighted_Renyi_Entropy',
            'Convergence_Status', 'Iterations', 'Time_sec'
        ]
        results_df = results_df.reindex(columns=[c for c in cols_order if c in results_df.columns])
        print("--- Individual Run Results Summary ---")
        print(results_df.to_string())
    
    if best_params:
        print(f"\n🏆 Best Parameters (from best single run Weighted NPMI): K={best_params['K']}, Eta={best_params['Eta']} with W_NPMI score of {best_single_run_npmi:.4f}")
    else:
        print("\n❌ No successful runs completed.")

    return results_df, best_model, best_params

In [12]:
# # ===== CTM 网格搜索实例 =====
# print("\n🎯 开始为 CTM 模型进行网格搜索以优化超参数...")
# print("=" * 60)

# # 1. 定义需要测试的超参数范围
# k_values_to_test = [20,30,40,50]
# eta_values_to_test = [0.01, 0.03, 0.05, 0.08, 0.10]

# # 2. 确保您的预处理文档已准备就绪
# ctm_docs = abstract_list 
# ALPHA = 0.1

# # 3. 运行网格搜索函数
# #    - num_runs=3: 每个参数组合运行3次
# #    - max_iters=500: 为了演示，减少迭代次数
# ctm_results_df, best_ctm_model, best_ctm_params = tune_ctm_model(
#     docs=ctm_docs,
#     k_range=k_values_to_test,
#     eta_range=eta_values_to_test,
#     ALPHA=ALPHA,
#     num_runs=5,
#     max_iters=500
# )

# # 4. 打印找到的最佳模型信息
# print("\n\n🎉 CTM 网格搜索完成！")
# if best_ctm_params:
#     # [修改] 更新打印信息，明确是基于 Weighted NPMI
#     print(f"🏆 找到的最佳超参数 (基于单次运行最高 Weighted NPMI): K={best_ctm_params.get('K')}, Eta={best_ctm_params.get('Eta')}")
    
#     # 您可以使用这个最佳模型进行进一步分析
#     print("\n🔍 对最佳CTM模型进行主题分析:")
#     analyze_model_topics(best_ctm_model, model_name="Best CTM", top_words=10)
# else:
#     print("❌ 未能找到最佳模型。")

# # 5. 显示并保存详细的、每一次运行的调优结果表格
# if ctm_results_df is not None and not ctm_results_df.empty:
#     print("\n--- CTM 最终全部运行调优结果 ---")
#     # [修改] 调整列顺序以获得更好的可读性
#     cols = [
#         'K', 'Eta', 'Run_Index', 'Weighted_NPMI', 'Perplexity', 'JSD_Diversity',
#         'Weighted_Cv', 'Unweighted_NPMI', 'Unweighted_Cv',
#         'Weighted_Renyi_Entropy', 'Unweighted_Renyi_Entropy',
#         'Convergence_Status', 'Iterations', 'Time_sec'
#     ]
#     display_df = ctm_results_df.reindex(columns=[c for c in cols if c in ctm_results_df.columns])
#     print(display_df.to_string())
    
#     # [修改] 更新保存路径和文件名
#     output_path = './data/model_result/ctm_full_grid_search_results.csv'
#     ctm_results_df.to_csv(output_path, index=False, encoding='utf-8-sig')
#     print(f"\n✅ 结果已保存到: {output_path}")


🎯 开始为 CTM 模型进行网格搜索以优化超参数...
🚀 Starting CTM Grid Search for 20 combinations, 5 runs each...
   (Convergence check: LL improvement < 1.0% for 10 consecutive checks of 10 iterations)

--- Combination 1/20: Testing K=20, Eta=0.01 for 5 runs ---
  - Run 1/5 (seed=42)...


/tmp/ipykernel_22137/3564434963.py:41: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(burn_in_iters)
/tmp/ipykernel_22137/3564434963.py:45: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(check_interval)


    => Converged after 300 iterations.
    => Results: PPL=163.47, W_NPMI=0.0036, JSD=0.9625, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=143.55, W_NPMI=0.0010, JSD=0.9787, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 iterations.
    => Results: PPL=162.14, W_NPMI=-0.0003, JSD=0.9693, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 iterations.
    => Results: PPL=149.08, W_NPMI=-0.0059, JSD=0.9758, Iters=300
  - Run 5/5 (seed=46)...
    => Converged after 300 iterations.
    => Results: PPL=167.40, W_NPMI=0.0032, JSD=0.9644, Iters=300

--- Combination 2/20: Testing K=20, Eta=0.03 for 5 runs ---
  - Run 1/5 (seed=42)...
    => Converged after 300 iterations.
    => Results: PPL=184.46, W_NPMI=0.0111, JSD=0.9694, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=190.16, W_NPMI=0.0087, JSD=0.9741, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 iterati

/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [-0.104035, -0.104048]


    => Converged after 300 iterations.
    => Results: PPL=686.42, W_NPMI=0.0183, JSD=0.9780, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=694.96, W_NPMI=0.0122, JSD=0.9770, Iters=300
  - Run 3/5 (seed=44)...


/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [0.0265795, 0.0259962]


    => Converged after 300 iterations.
    => Results: PPL=705.80, W_NPMI=0.0192, JSD=0.9783, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 iterations.
    => Results: PPL=796.44, W_NPMI=0.0309, JSD=0.9697, Iters=300
  - Run 5/5 (seed=46)...
    => Converged after 300 iterations.
    => Results: PPL=654.00, W_NPMI=0.0099, JSD=0.9790, Iters=300


✅ CTM Tuning Complete!
--- Individual Run Results Summary ---
     K   Eta  Run_Index  Weighted_NPMI  Perplexity  JSD_Diversity  Weighted_Cv  Unweighted_NPMI  Unweighted_Cv  Weighted_Renyi_Entropy  Unweighted_Renyi_Entropy Convergence_Status  Iterations   Time_sec
0   20  0.01          1       0.003580  163.465147       0.962493     0.407931         0.001410       0.429173                3.602872                  3.622156          Converged         300  31.023284
1   20  0.01          2       0.001043  143.546217       0.978707     0.421363         0.005553       0.463601                3.480325                  3.518048        

In [12]:
# ===== CTM 网格搜索实例 =====
print("\n🎯 开始为 CTM 模型进行网格搜索以优化超参数...")
print("=" * 60)

# 1. 定义需要测试的超参数范围
k_values_to_test = [60]
eta_values_to_test = [0.01, 0.03, 0.05, 0.08, 0.10]

# 2. 确保您的预处理文档已准备就绪
ctm_docs = abstract_list 
ALPHA = 0.1

# 3. 运行网格搜索函数
#    - num_runs=3: 每个参数组合运行3次
#    - max_iters=500: 为了演示，减少迭代次数
ctm_results_df, best_ctm_model, best_ctm_params = tune_ctm_model(
    docs=ctm_docs,
    k_range=k_values_to_test,
    eta_range=eta_values_to_test,
    ALPHA=ALPHA,
    num_runs=5,
    max_iters=500
)

# 4. 打印找到的最佳模型信息
print("\n\n🎉 CTM 网格搜索完成！")
if best_ctm_params:
    # [修改] 更新打印信息，明确是基于 Weighted NPMI
    print(f"🏆 找到的最佳超参数 (基于单次运行最高 Weighted NPMI): K={best_ctm_params.get('K')}, Eta={best_ctm_params.get('Eta')}")
    
    # 您可以使用这个最佳模型进行进一步分析
    print("\n🔍 对最佳CTM模型进行主题分析:")
    analyze_model_topics(best_ctm_model, model_name="Best CTM", top_words=10)
else:
    print("❌ 未能找到最佳模型。")

# 5. 显示并保存详细的、每一次运行的调优结果表格
if ctm_results_df is not None and not ctm_results_df.empty:
    print("\n--- CTM 最终全部运行调优结果 ---")
    # [修改] 调整列顺序以获得更好的可读性
    cols = [
        'K', 'Eta', 'Run_Index', 'Weighted_NPMI', 'Perplexity', 'JSD_Diversity',
        'Weighted_Cv', 'Unweighted_NPMI', 'Unweighted_Cv',
        'Weighted_Renyi_Entropy', 'Unweighted_Renyi_Entropy',
        'Convergence_Status', 'Iterations', 'Time_sec'
    ]
    display_df = ctm_results_df.reindex(columns=[c for c in cols if c in ctm_results_df.columns])
    print(display_df.to_string())
    
    # [修改] 更新保存路径和文件名
    output_path = './data/model_result/ctm_full_grid_search_results_added.csv'
    ctm_results_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 结果已保存到: {output_path}")


🎯 开始为 CTM 模型进行网格搜索以优化超参数...
🚀 Starting CTM Grid Search for 5 combinations, 5 runs each...
   (Convergence check: LL improvement < 1.0% for 10 consecutive checks of 10 iterations)

--- Combination 1/5: Testing K=60, Eta=0.01 for 5 runs ---
  - Run 1/5 (seed=42)...


/tmp/ipykernel_52815/3564434963.py:41: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(burn_in_iters)
/tmp/ipykernel_52815/3564434963.py:45: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(check_interval)


    => Converged after 300 iterations.
    => Results: PPL=480.83, W_NPMI=0.0080, JSD=0.9754, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=519.39, W_NPMI=-0.0103, JSD=0.9729, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 iterations.
    => Results: PPL=465.62, W_NPMI=-0.0025, JSD=0.9765, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 iterations.
    => Results: PPL=471.12, W_NPMI=-0.0105, JSD=0.9763, Iters=300
  - Run 5/5 (seed=46)...
    => Converged after 300 iterations.
    => Results: PPL=453.73, W_NPMI=-0.0068, JSD=0.9761, Iters=300

--- Combination 2/5: Testing K=60, Eta=0.03 for 5 runs ---
  - Run 1/5 (seed=42)...
    => Converged after 300 iterations.
    => Results: PPL=611.58, W_NPMI=-0.0005, JSD=0.9769, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=636.54, W_NPMI=0.0067, JSD=0.9742, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 itera

/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [0.25796, 0.221879]


    => Converged after 300 iterations.
    => Results: PPL=844.87, W_NPMI=0.0090, JSD=0.9795, Iters=300

--- Combination 5/5: Testing K=60, Eta=0.1 for 5 runs ---
  - Run 1/5 (seed=42)...
    => Converged after 300 iterations.
    => Results: PPL=964.39, W_NPMI=0.0049, JSD=0.9781, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=883.75, W_NPMI=0.0138, JSD=0.9799, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 iterations.
    => Results: PPL=998.64, W_NPMI=-0.0004, JSD=0.9761, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 iterations.
    => Results: PPL=903.96, W_NPMI=0.0142, JSD=0.9788, Iters=300
  - Run 5/5 (seed=46)...
    => Converged after 300 iterations.
    => Results: PPL=978.72, W_NPMI=0.0141, JSD=0.9790, Iters=300


✅ CTM Tuning Complete!
--- Individual Run Results Summary ---
     K   Eta  Run_Index  Weighted_NPMI  Perplexity  JSD_Diversity  Weighted_Cv  Unweighted_NPMI  Unweighted_Cv  Weighted_Renyi_

In [12]:
# ===== CTM 网格搜索实例 =====
print("\n🎯 开始为 CTM 模型进行网格搜索以优化超参数...")
print("=" * 60)

# 1. 定义需要测试的超参数范围
k_values_to_test = [30]
eta_values_to_test = [1,2]

# 2. 确保您的预处理文档已准备就绪
ctm_docs = abstract_list 
ALPHA = 0.1

# 3. 运行网格搜索函数
#    - num_runs=3: 每个参数组合运行3次
#    - max_iters=500: 为了演示，减少迭代次数
ctm_results_df, best_ctm_model, best_ctm_params = tune_ctm_model(
    docs=ctm_docs,
    k_range=k_values_to_test,
    eta_range=eta_values_to_test,
    ALPHA=ALPHA,
    num_runs=5,
    max_iters=500
)

# 4. 打印找到的最佳模型信息
print("\n\n🎉 CTM 网格搜索完成！")
if best_ctm_params:
    # [修改] 更新打印信息，明确是基于 Weighted NPMI
    print(f"🏆 找到的最佳超参数 (基于单次运行最高 Weighted NPMI): K={best_ctm_params.get('K')}, Eta={best_ctm_params.get('Eta')}")
    
    # 您可以使用这个最佳模型进行进一步分析
    print("\n🔍 对最佳CTM模型进行主题分析:")
    analyze_model_topics(best_ctm_model, model_name="Best CTM", top_words=10)
else:
    print("❌ 未能找到最佳模型。")

# 5. 显示并保存详细的、每一次运行的调优结果表格
if ctm_results_df is not None and not ctm_results_df.empty:
    print("\n--- CTM 最终全部运行调优结果 ---")
    # [修改] 调整列顺序以获得更好的可读性
    cols = [
        'K', 'Eta', 'Run_Index', 'Weighted_NPMI', 'Perplexity', 'JSD_Diversity',
        'Weighted_Cv', 'Unweighted_NPMI', 'Unweighted_Cv',
        'Weighted_Renyi_Entropy', 'Unweighted_Renyi_Entropy',
        'Convergence_Status', 'Iterations', 'Time_sec'
    ]
    display_df = ctm_results_df.reindex(columns=[c for c in cols if c in ctm_results_df.columns])
    print(display_df.to_string())
    
    # [修改] 更新保存路径和文件名
    output_path = './data/model_result/ctm_full_grid_search_results_added_2.csv'
    ctm_results_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 结果已保存到: {output_path}")


🎯 开始为 CTM 模型进行网格搜索以优化超参数...
🚀 Starting CTM Grid Search for 2 combinations, 5 runs each...
   (Convergence check: LL improvement < 1.0% for 10 consecutive checks of 10 iterations)

--- Combination 1/2: Testing K=30, Eta=1 for 5 runs ---
  - Run 1/5 (seed=42)...


/tmp/ipykernel_1104/3564434963.py:41: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(burn_in_iters)
/tmp/ipykernel_1104/3564434963.py:45: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(check_interval)


    => Converged after 300 iterations.
    => Results: PPL=1076.15, W_NPMI=0.0053, JSD=0.9775, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=1053.56, W_NPMI=-0.0039, JSD=0.9779, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 iterations.
    => Results: PPL=1072.63, W_NPMI=-0.0019, JSD=0.9751, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 iterations.
    => Results: PPL=1101.78, W_NPMI=0.0027, JSD=0.9749, Iters=300
  - Run 5/5 (seed=46)...
    => Converged after 300 iterations.
    => Results: PPL=1056.14, W_NPMI=-0.0042, JSD=0.9824, Iters=300

--- Combination 2/2: Testing K=30, Eta=2 for 5 runs ---
  - Run 1/5 (seed=42)...
    => Converged after 300 iterations.
    => Results: PPL=1607.80, W_NPMI=-0.0233, JSD=0.9388, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=1538.23, W_NPMI=-0.0159, JSD=0.9505, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 i

In [14]:
# ===== CTM 网格搜索实例 =====
print("\n🎯 开始为 CTM 模型进行网格搜索以优化超参数...")
print("=" * 60)

# 1. 定义需要测试的超参数范围
k_values_to_test = [80]
eta_values_to_test = [0.01, 0.03, 0.05, 0.08, 0.10]

# 2. 确保您的预处理文档已准备就绪
ctm_docs = abstract_list 
ALPHA = 0.1

# 3. 运行网格搜索函数
#    - num_runs=3: 每个参数组合运行3次
#    - max_iters=500: 为了演示，减少迭代次数
ctm_results_df, best_ctm_model, best_ctm_params = tune_ctm_model(
    docs=ctm_docs,
    k_range=k_values_to_test,
    eta_range=eta_values_to_test,
    ALPHA=ALPHA,
    num_runs=5,
    max_iters=500
)

# 4. 打印找到的最佳模型信息
print("\n\n🎉 CTM 网格搜索完成！")
if best_ctm_params:
    # [修改] 更新打印信息，明确是基于 Weighted NPMI
    print(f"🏆 找到的最佳超参数 (基于单次运行最高 Weighted NPMI): K={best_ctm_params.get('K')}, Eta={best_ctm_params.get('Eta')}")
    
    # 您可以使用这个最佳模型进行进一步分析
    print("\n🔍 对最佳CTM模型进行主题分析:")
    analyze_model_topics(best_ctm_model, model_name="Best CTM", top_words=10)
else:
    print("❌ 未能找到最佳模型。")

# 5. 显示并保存详细的、每一次运行的调优结果表格
if ctm_results_df is not None and not ctm_results_df.empty:
    print("\n--- CTM 最终全部运行调优结果 ---")
    # [修改] 调整列顺序以获得更好的可读性
    cols = [
        'K', 'Eta', 'Run_Index', 'Weighted_NPMI', 'Perplexity', 'JSD_Diversity',
        'Weighted_Cv', 'Unweighted_NPMI', 'Unweighted_Cv',
        'Weighted_Renyi_Entropy', 'Unweighted_Renyi_Entropy',
        'Convergence_Status', 'Iterations', 'Time_sec'
    ]
    display_df = ctm_results_df.reindex(columns=[c for c in cols if c in ctm_results_df.columns])
    print(display_df.to_string())
    
    # [修改] 更新保存路径和文件名
    output_path = './data/model_result/ctm_full_grid_search_results_added_3.csv'
    ctm_results_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 结果已保存到: {output_path}")


🎯 开始为 CTM 模型进行网格搜索以优化超参数...
🚀 Starting CTM Grid Search for 5 combinations, 5 runs each...
   (Convergence check: LL improvement < 1.0% for 10 consecutive checks of 10 iterations)

--- Combination 1/5: Testing K=80, Eta=0.01 for 5 runs ---
  - Run 1/5 (seed=42)...


/tmp/ipykernel_52815/3564434963.py:41: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(burn_in_iters)
/tmp/ipykernel_52815/3564434963.py:45: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(check_interval)


    => Converged after 300 iterations.
    => Results: PPL=772.88, W_NPMI=-0.0065, JSD=0.9776, Iters=300
  - Run 2/5 (seed=43)...


/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [2.37839, 2.37804]


    => Converged after 300 iterations.
    => Results: PPL=836.68, W_NPMI=-0.0111, JSD=0.9788, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 iterations.
    => Results: PPL=825.71, W_NPMI=-0.0105, JSD=0.9797, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 iterations.
    => Results: PPL=806.34, W_NPMI=-0.0067, JSD=0.9781, Iters=300
  - Run 5/5 (seed=46)...
    => Converged after 300 iterations.
    => Results: PPL=798.63, W_NPMI=-0.0091, JSD=0.9793, Iters=300

--- Combination 2/5: Testing K=80, Eta=0.03 for 5 runs ---
  - Run 1/5 (seed=42)...
    => Converged after 300 iterations.
    => Results: PPL=1147.68, W_NPMI=-0.0017, JSD=0.9764, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=1080.87, W_NPMI=0.0028, JSD=0.9787, Iters=300
  - Run 3/5 (seed=44)...


/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [-1.09148, -1.09148]


    => Converged after 300 iterations.
    => Results: PPL=1146.28, W_NPMI=0.0051, JSD=0.9779, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 iterations.
    => Results: PPL=1112.53, W_NPMI=-0.0050, JSD=0.9787, Iters=300
  - Run 5/5 (seed=46)...
    => Converged after 300 iterations.
    => Results: PPL=1117.31, W_NPMI=0.0147, JSD=0.9788, Iters=300

--- Combination 3/5: Testing K=80, Eta=0.05 for 5 runs ---
  - Run 1/5 (seed=42)...


/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [1.14313, 0.570381]
/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [-0.770424, -10.1537]
/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [473.65, -105.877]
/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [2569.67, -40237]
/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [9.26729e+06, -451498]
/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [7.37816e+07, -4.71697e+07]
/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [4.82674e+08, -1.83704e+09]
/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [6.92915e+09, -3.42753e+10]
/__w/tomotopy/tomotopy/src/TopicModel/../Utils

    => Converged after 300 iterations.
    => Results: PPL=1302.52, W_NPMI=0.0067, JSD=0.9808, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=1326.52, W_NPMI=0.0094, JSD=0.9797, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 iterations.
    => Results: PPL=1384.07, W_NPMI=0.0026, JSD=0.9780, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 iterations.
    => Results: PPL=1356.49, W_NPMI=-0.0005, JSD=0.9791, Iters=300
  - Run 5/5 (seed=46)...
    => Converged after 300 iterations.
    => Results: PPL=1444.03, W_NPMI=0.0065, JSD=0.9784, Iters=300

--- Combination 4/5: Testing K=80, Eta=0.08 for 5 runs ---
  - Run 1/5 (seed=42)...
    => Converged after 300 iterations.
    => Results: PPL=1652.10, W_NPMI=0.0075, JSD=0.9791, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=1644.87, W_NPMI=0.0041, JSD=0.9805, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 it

In [15]:
# ===== CTM 网格搜索实例 =====
print("\n🎯 开始为 CTM 模型进行网格搜索以优化超参数...")
print("=" * 60)

# 1. 定义需要测试的超参数范围
k_values_to_test = [90]
eta_values_to_test = [0.01, 0.03, 0.05, 0.08, 0.10]

# 2. 确保您的预处理文档已准备就绪
ctm_docs = abstract_list 
ALPHA = 0.1

# 3. 运行网格搜索函数
#    - num_runs=3: 每个参数组合运行3次
#    - max_iters=500: 为了演示，减少迭代次数
ctm_results_df, best_ctm_model, best_ctm_params = tune_ctm_model(
    docs=ctm_docs,
    k_range=k_values_to_test,
    eta_range=eta_values_to_test,
    ALPHA=ALPHA,
    num_runs=5,
    max_iters=500
)

# 4. 打印找到的最佳模型信息
print("\n\n🎉 CTM 网格搜索完成！")
if best_ctm_params:
    # [修改] 更新打印信息，明确是基于 Weighted NPMI
    print(f"🏆 找到的最佳超参数 (基于单次运行最高 Weighted NPMI): K={best_ctm_params.get('K')}, Eta={best_ctm_params.get('Eta')}")
    
    # 您可以使用这个最佳模型进行进一步分析
    print("\n🔍 对最佳CTM模型进行主题分析:")
    analyze_model_topics(best_ctm_model, model_name="Best CTM", top_words=10)
else:
    print("❌ 未能找到最佳模型。")

# 5. 显示并保存详细的、每一次运行的调优结果表格
if ctm_results_df is not None and not ctm_results_df.empty:
    print("\n--- CTM 最终全部运行调优结果 ---")
    # [修改] 调整列顺序以获得更好的可读性
    cols = [
        'K', 'Eta', 'Run_Index', 'Weighted_NPMI', 'Perplexity', 'JSD_Diversity',
        'Weighted_Cv', 'Unweighted_NPMI', 'Unweighted_Cv',
        'Weighted_Renyi_Entropy', 'Unweighted_Renyi_Entropy',
        'Convergence_Status', 'Iterations', 'Time_sec'
    ]
    display_df = ctm_results_df.reindex(columns=[c for c in cols if c in ctm_results_df.columns])
    print(display_df.to_string())
    
    # [修改] 更新保存路径和文件名
    output_path = './data/model_result/ctm_full_grid_search_results_added_4.csv'
    ctm_results_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 结果已保存到: {output_path}")


🎯 开始为 CTM 模型进行网格搜索以优化超参数...
🚀 Starting CTM Grid Search for 5 combinations, 5 runs each...
   (Convergence check: LL improvement < 1.0% for 10 consecutive checks of 10 iterations)

--- Combination 1/5: Testing K=90, Eta=0.01 for 5 runs ---
  - Run 1/5 (seed=42)...


/tmp/ipykernel_52815/3564434963.py:41: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(burn_in_iters)
/tmp/ipykernel_52815/3564434963.py:45: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(check_interval)


    => Converged after 300 iterations.
    => Results: PPL=1083.09, W_NPMI=-0.0202, JSD=0.9799, Iters=300
  - Run 2/5 (seed=43)...


/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [-0.288958, -0.300655]


    => Converged after 300 iterations.
    => Results: PPL=1090.08, W_NPMI=-0.0167, JSD=0.9802, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 iterations.
    => Results: PPL=1127.87, W_NPMI=-0.0109, JSD=0.9769, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 iterations.
    => Results: PPL=1105.43, W_NPMI=-0.0199, JSD=0.9798, Iters=300
  - Run 5/5 (seed=46)...
    => Converged after 300 iterations.
    => Results: PPL=1058.34, W_NPMI=-0.0187, JSD=0.9798, Iters=300

--- Combination 2/5: Testing K=90, Eta=0.03 for 5 runs ---
  - Run 1/5 (seed=42)...


...

    => Converged after 300 iterations.
    => Results: PPL=1726.10, W_NPMI=-0.0004, JSD=0.9818, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=1827.85, W_NPMI=0.0010, JSD=0.9791, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 iterations.
    => Results: PPL=1699.21, W_NPMI=0.0016, JSD=0.9810, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 iterations.
    => Results: PPL=1785.58, W_NPMI=0.0027, JSD=0.9801, Iters=300
  - Run 5/5 (seed=46)...
    => Converged after 300 iterations.
    => Results: PPL=1808.50, W_NPMI=0.0079, JSD=0.9789, Iters=300

--- Combination 4/5: Testing K=90, Eta=0.08 for 5 runs ---
  - Run 1/5 (seed=42)...
    => Converged after 300 iterations.
    => Results: PPL=2274.91, W_NPMI=0.0081, JSD=0.9802, Iters=300
  - Run 2/5 (seed=43)...


/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [2.23042, 2.2304]


    => Converged after 300 iterations.
    => Results: PPL=2155.43, W_NPMI=-0.0028, JSD=0.9818, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 iterations.
    => Results: PPL=2065.61, W_NPMI=0.0075, JSD=0.9823, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 iterations.
    => Results: PPL=1966.74, W_NPMI=0.0075, JSD=0.9830, Iters=300
  - Run 5/5 (seed=46)...
    => Converged after 300 iterations.
    => Results: PPL=1984.25, W_NPMI=0.0052, JSD=0.9833, Iters=300

--- Combination 5/5: Testing K=90, Eta=0.1 for 5 runs ---
  - Run 1/5 (seed=42)...


/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [-0.269431, -0.269433]
/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [0.878475, 0.878473]


    => Converged after 300 iterations.
    => Results: PPL=2439.82, W_NPMI=0.0016, JSD=0.9823, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=2337.27, W_NPMI=-0.0061, JSD=0.9826, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 iterations.
    => Results: PPL=2206.18, W_NPMI=0.0073, JSD=0.9830, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 iterations.
    => Results: PPL=2539.34, W_NPMI=0.0042, JSD=0.9816, Iters=300
  - Run 5/5 (seed=46)...
    => Converged after 300 iterations.
    => Results: PPL=2296.75, W_NPMI=0.0112, JSD=0.9830, Iters=300


✅ CTM Tuning Complete!
--- Individual Run Results Summary ---
     K   Eta  Run_Index  Weighted_NPMI   Perplexity  JSD_Diversity  Weighted_Cv  Unweighted_NPMI  Unweighted_Cv  Weighted_Renyi_Entropy  Unweighted_Renyi_Entropy Convergence_Status  Iterations    Time_sec
0   90  0.01          1      -0.020160  1083.091034       0.979942     0.444463        -0.043442       0.

In [12]:
# ===== CTM 网格搜索实例 =====
print("\n🎯 开始为 CTM 模型进行网格搜索以优化超参数...")
print("=" * 60)

# 1. 定义需要测试的超参数范围
k_values_to_test = [100]
eta_values_to_test = [0.01, 0.03]

# 2. 确保您的预处理文档已准备就绪
ctm_docs = abstract_list 
ALPHA = 0.1

# 3. 运行网格搜索函数
#    - num_runs=3: 每个参数组合运行3次
#    - max_iters=500: 为了演示，减少迭代次数
ctm_results_df, best_ctm_model, best_ctm_params = tune_ctm_model(
    docs=ctm_docs,
    k_range=k_values_to_test,
    eta_range=eta_values_to_test,
    ALPHA=ALPHA,
    num_runs=5,
    max_iters=500
)

# 4. 打印找到的最佳模型信息
print("\n\n🎉 CTM 网格搜索完成！")
if best_ctm_params:
    # [修改] 更新打印信息，明确是基于 Weighted NPMI
    print(f"🏆 找到的最佳超参数 (基于单次运行最高 Weighted NPMI): K={best_ctm_params.get('K')}, Eta={best_ctm_params.get('Eta')}")
    
    # 您可以使用这个最佳模型进行进一步分析
    print("\n🔍 对最佳CTM模型进行主题分析:")
    analyze_model_topics(best_ctm_model, model_name="Best CTM", top_words=10)
else:
    print("❌ 未能找到最佳模型。")

# 5. 显示并保存详细的、每一次运行的调优结果表格
if ctm_results_df is not None and not ctm_results_df.empty:
    print("\n--- CTM 最终全部运行调优结果 ---")
    # [修改] 调整列顺序以获得更好的可读性
    cols = [
        'K', 'Eta', 'Run_Index', 'Weighted_NPMI', 'Perplexity', 'JSD_Diversity',
        'Weighted_Cv', 'Unweighted_NPMI', 'Unweighted_Cv',
        'Weighted_Renyi_Entropy', 'Unweighted_Renyi_Entropy',
        'Convergence_Status', 'Iterations', 'Time_sec'
    ]
    display_df = ctm_results_df.reindex(columns=[c for c in cols if c in ctm_results_df.columns])
    print(display_df.to_string())
    
    # [修改] 更新保存路径和文件名
    output_path = './data/model_result/ctm_full_grid_search_results_added_5.csv'
    ctm_results_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 结果已保存到: {output_path}")


🎯 开始为 CTM 模型进行网格搜索以优化超参数...
🚀 Starting CTM Grid Search for 2 combinations, 5 runs each...
   (Convergence check: LL improvement < 1.0% for 10 consecutive checks of 10 iterations)

--- Combination 1/2: Testing K=100, Eta=0.01 for 5 runs ---
  - Run 1/5 (seed=42)...


/tmp/ipykernel_4287/3564434963.py:41: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(burn_in_iters)
/tmp/ipykernel_4287/3564434963.py:45: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(check_interval)


    => Converged after 300 iterations.
    => Results: PPL=1384.72, W_NPMI=-0.0220, JSD=0.9817, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=1394.57, W_NPMI=-0.0186, JSD=0.9810, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 iterations.
    => Results: PPL=1444.70, W_NPMI=-0.0180, JSD=0.9805, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 iterations.
    => Results: PPL=1429.64, W_NPMI=-0.0241, JSD=0.9797, Iters=300
  - Run 5/5 (seed=46)...
    => Converged after 300 iterations.
    => Results: PPL=1374.53, W_NPMI=-0.0225, JSD=0.9811, Iters=300

--- Combination 2/2: Testing K=100, Eta=0.03 for 5 runs ---
  - Run 1/5 (seed=42)...
    => Converged after 300 iterations.
    => Results: PPL=1966.09, W_NPMI=-0.0039, JSD=0.9804, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=1873.67, W_NPMI=-0.0034, JSD=0.9826, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after

In [13]:
# ===== CTM 网格搜索实例 =====
print("\n🎯 开始为 CTM 模型进行网格搜索以优化超参数...")
print("=" * 60)

# 1. 定义需要测试的超参数范围
k_values_to_test = [100]
eta_values_to_test = [0.05, 0.08]

# 2. 确保您的预处理文档已准备就绪
ctm_docs = abstract_list 
ALPHA = 0.1

# 3. 运行网格搜索函数
#    - num_runs=3: 每个参数组合运行3次
#    - max_iters=500: 为了演示，减少迭代次数
ctm_results_df, best_ctm_model, best_ctm_params = tune_ctm_model(
    docs=ctm_docs,
    k_range=k_values_to_test,
    eta_range=eta_values_to_test,
    ALPHA=ALPHA,
    num_runs=5,
    max_iters=500
)

# 4. 打印找到的最佳模型信息
print("\n\n🎉 CTM 网格搜索完成！")
if best_ctm_params:
    # [修改] 更新打印信息，明确是基于 Weighted NPMI
    print(f"🏆 找到的最佳超参数 (基于单次运行最高 Weighted NPMI): K={best_ctm_params.get('K')}, Eta={best_ctm_params.get('Eta')}")
    
    # 您可以使用这个最佳模型进行进一步分析
    print("\n🔍 对最佳CTM模型进行主题分析:")
    analyze_model_topics(best_ctm_model, model_name="Best CTM", top_words=10)
else:
    print("❌ 未能找到最佳模型。")

# 5. 显示并保存详细的、每一次运行的调优结果表格
if ctm_results_df is not None and not ctm_results_df.empty:
    print("\n--- CTM 最终全部运行调优结果 ---")
    # [修改] 调整列顺序以获得更好的可读性
    cols = [
        'K', 'Eta', 'Run_Index', 'Weighted_NPMI', 'Perplexity', 'JSD_Diversity',
        'Weighted_Cv', 'Unweighted_NPMI', 'Unweighted_Cv',
        'Weighted_Renyi_Entropy', 'Unweighted_Renyi_Entropy',
        'Convergence_Status', 'Iterations', 'Time_sec'
    ]
    display_df = ctm_results_df.reindex(columns=[c for c in cols if c in ctm_results_df.columns])
    print(display_df.to_string())
    
    # [修改] 更新保存路径和文件名
    output_path = './data/model_result/ctm_full_grid_search_results_added_6.csv'
    ctm_results_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 结果已保存到: {output_path}")


🎯 开始为 CTM 模型进行网格搜索以优化超参数...
🚀 Starting CTM Grid Search for 2 combinations, 5 runs each...
   (Convergence check: LL improvement < 1.0% for 10 consecutive checks of 10 iterations)

--- Combination 1/2: Testing K=100, Eta=0.05 for 5 runs ---
  - Run 1/5 (seed=42)...


/tmp/ipykernel_4287/3564434963.py:41: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(burn_in_iters)
/tmp/ipykernel_4287/3564434963.py:45: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(check_interval)


    => Converged after 300 iterations.
    => Results: PPL=2460.63, W_NPMI=-0.0055, JSD=0.9804, Iters=300
  - Run 2/5 (seed=43)...


/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [1.23942, 1.20457]


    => Converged after 300 iterations.
    => Results: PPL=2521.23, W_NPMI=0.0088, JSD=0.9795, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 iterations.
    => Results: PPL=2192.13, W_NPMI=0.0028, JSD=0.9834, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 iterations.
    => Results: PPL=2337.19, W_NPMI=0.0010, JSD=0.9818, Iters=300
  - Run 5/5 (seed=46)...
    => Converged after 300 iterations.
    => Results: PPL=2530.31, W_NPMI=-0.0078, JSD=0.9808, Iters=300

--- Combination 2/2: Testing K=100, Eta=0.08 for 5 runs ---
  - Run 1/5 (seed=42)...
    => Converged after 300 iterations.
    => Results: PPL=2743.09, W_NPMI=0.0100, JSD=0.9835, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=2951.70, W_NPMI=0.0039, JSD=0.9827, Iters=300
  - Run 3/5 (seed=44)...
    => Converged after 300 iterations.
    => Results: PPL=2926.56, W_NPMI=0.0046, JSD=0.9811, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 i

In [14]:
# ===== CTM 网格搜索实例 =====
print("\n🎯 开始为 CTM 模型进行网格搜索以优化超参数...")
print("=" * 60)

# 1. 定义需要测试的超参数范围
k_values_to_test = [100]
eta_values_to_test = [0.10]

# 2. 确保您的预处理文档已准备就绪
ctm_docs = abstract_list 
ALPHA = 0.1

# 3. 运行网格搜索函数
#    - num_runs=3: 每个参数组合运行3次
#    - max_iters=500: 为了演示，减少迭代次数
ctm_results_df, best_ctm_model, best_ctm_params = tune_ctm_model(
    docs=ctm_docs,
    k_range=k_values_to_test,
    eta_range=eta_values_to_test,
    ALPHA=ALPHA,
    num_runs=5,
    max_iters=500
)

# 4. 打印找到的最佳模型信息
print("\n\n🎉 CTM 网格搜索完成！")
if best_ctm_params:
    # [修改] 更新打印信息，明确是基于 Weighted NPMI
    print(f"🏆 找到的最佳超参数 (基于单次运行最高 Weighted NPMI): K={best_ctm_params.get('K')}, Eta={best_ctm_params.get('Eta')}")
    
    # 您可以使用这个最佳模型进行进一步分析
    print("\n🔍 对最佳CTM模型进行主题分析:")
    analyze_model_topics(best_ctm_model, model_name="Best CTM", top_words=10)
else:
    print("❌ 未能找到最佳模型。")

# 5. 显示并保存详细的、每一次运行的调优结果表格
if ctm_results_df is not None and not ctm_results_df.empty:
    print("\n--- CTM 最终全部运行调优结果 ---")
    # [修改] 调整列顺序以获得更好的可读性
    cols = [
        'K', 'Eta', 'Run_Index', 'Weighted_NPMI', 'Perplexity', 'JSD_Diversity',
        'Weighted_Cv', 'Unweighted_NPMI', 'Unweighted_Cv',
        'Weighted_Renyi_Entropy', 'Unweighted_Renyi_Entropy',
        'Convergence_Status', 'Iterations', 'Time_sec'
    ]
    display_df = ctm_results_df.reindex(columns=[c for c in cols if c in ctm_results_df.columns])
    print(display_df.to_string())
    
    # [修改] 更新保存路径和文件名
    output_path = './data/model_result/ctm_full_grid_search_results_added_7.csv'
    ctm_results_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 结果已保存到: {output_path}")


🎯 开始为 CTM 模型进行网格搜索以优化超参数...
🚀 Starting CTM Grid Search for 1 combinations, 5 runs each...
   (Convergence check: LL improvement < 1.0% for 10 consecutive checks of 10 iterations)

--- Combination 1/1: Testing K=100, Eta=0.1 for 5 runs ---
  - Run 1/5 (seed=42)...


/tmp/ipykernel_4287/3564434963.py:41: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(burn_in_iters)
/tmp/ipykernel_4287/3564434963.py:45: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(check_interval)
/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [3.04328, 3.04328]
/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [0.857147, 0.854964]
/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [0.409063, 0.398998]


    => Converged after 300 iterations.
    => Results: PPL=3160.99, W_NPMI=0.0049, JSD=0.9831, Iters=300
  - Run 2/5 (seed=43)...
    => Converged after 300 iterations.
    => Results: PPL=3240.69, W_NPMI=0.0060, JSD=0.9826, Iters=300
  - Run 3/5 (seed=44)...


/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [0.510531, 0.510318]
/__w/tomotopy/tomotopy/src/TopicModel/../Utils/TruncMultiNormal.hpp(56): wrong truncation range [0.744091, 0.744085]


    => Converged after 300 iterations.
    => Results: PPL=3365.00, W_NPMI=0.0125, JSD=0.9820, Iters=300
  - Run 4/5 (seed=45)...
    => Converged after 300 iterations.
    => Results: PPL=3105.64, W_NPMI=0.0104, JSD=0.9833, Iters=300
  - Run 5/5 (seed=46)...
    => Converged after 300 iterations.
    => Results: PPL=2997.29, W_NPMI=0.0057, JSD=0.9833, Iters=300


✅ CTM Tuning Complete!
--- Individual Run Results Summary ---
     K  Eta  Run_Index  Weighted_NPMI   Perplexity  JSD_Diversity  Weighted_Cv  Unweighted_NPMI  Unweighted_Cv  Weighted_Renyi_Entropy  Unweighted_Renyi_Entropy Convergence_Status  Iterations    Time_sec
0  100  0.1          1       0.004884  3160.986468       0.983076     0.479292        -0.042422       0.484408                3.662238                  3.870220          Converged         300  166.891760
1  100  0.1          2       0.005987  3240.685175       0.982637     0.484720        -0.042524       0.487454                3.663171                  3.872809   